# Final Project — Merge FF3 Factors
**Business 35137 — Winter 2026**

Pulls Fama-French 3-factor monthly data from CRSP/WRDS and merges onto `panel.parquet` on `date`.

In [ ]:
import pandas as pd
import wrds
import os

DATA_DIR = '../Data/'
intermediate_dir = os.path.join(DATA_DIR, 'Intermediate')

panel = pd.read_parquet(os.path.join(intermediate_dir, 'panel.parquet'))
print(f"Panel shape: {panel.shape}")
print(f"Date range: {panel['date'].min():%Y-%m} – {panel['date'].max():%Y-%m}")

In [6]:
# Cell 2 — Pull FF3 from WRDS
db = wrds.Connection(wrds_username=os.environ.get('WRDS_USERNAME'))  # set WRDS_USERNAME env var

ff3 = db.raw_sql("""
    SELECT date, mktrf, smb, hml, rf
    FROM ff.factors_monthly
""", date_cols=['date'])

db.close()

ff3['ym'] = ff3['date'].dt.to_period('M')

print(f"FF3 rows: {len(ff3):,}")
print(f"Date range: {ff3['date'].min():%Y-%m} – {ff3['date'].max():%Y-%m}")
ff3.head()

Loading library list...
Done
FF3 rows: 1,194
Date range: 1926-07 – 2025-12


,date,mktrf,smb,hml,rf,ym
0,1926-07-01,0.0289,-0.0255,-0.0239,0.0022,1926-07
1,1926-08-01,0.0264,-0.0114,0.0381,0.0025,1926-08
2,1926-09-01,0.0038,-0.0136,0.0005,0.0023,1926-09
3,1926-10-01,-0.0327,-0.0014,0.0082,0.0032,1926-10
4,1926-11-01,0.0254,-0.0011,-0.0061,0.0031,1926-11


In [7]:

panel['ym'] = panel['date'].dt.to_period('M')

panel = panel.merge(ff3[['ym', 'mktrf', 'smb', 'hml', 'rf']], on='ym', how='left')
panel.drop(columns='ym', inplace=True)

n_matched = panel['mktrf'].notna().sum()
print(f"Matched: {n_matched:,} / {len(panel):,}  ({n_matched/len(panel):.1%})")
panel[['date', 'mktrf', 'smb', 'hml', 'rf']].dropna().head()



Matched: 215,342 / 215,342  (100.0%)


,date,mktrf,smb,hml,rf
0,2011-04-29,0.029,-0.0042,-0.0244,0.0
1,2011-05-31,-0.0127,-0.0073,-0.0185,0.0
2,2011-06-30,-0.0174,-0.0027,-0.002,0.0
3,2011-07-29,-0.0235,-0.0117,-0.0091,0.0
4,2011-08-31,-0.0596,-0.0296,-0.0243,0.0001


In [8]:
# Cell 4 — Save
panel.to_parquet(os.path.join(intermediate_dir, 'panel.parquet'), index=False)
print("Saved panel.parquet with FF3 factors.")

Saved panel.parquet with FF3 factors.
